In [93]:
import pandas as pd
df = pd.read_pickle('coef_cleaned_one.pkl')
df.head()

,insurance_type,label,prev_readmit_group,los_group,followup_visits_group,risk_score_bin,dc_location,primary_dx_tier,age_bin
0,Medicare/Medicaid,1,1,6 to 8,5 or more,7,HH/SNF,lower,2.0
1,Private,1,2,6 to 8,4,7,HH/SNF,lower,2.0
2,Medicare/Medicaid,1,2,6 to 8,5 or more,7,HH/SNF,lower,3.0
3,Medicare/Medicaid,1,2,9+,4,7,HH/SNF,higher,3.0
4,Uninsured,1,1,9+,2,6,HH/SNF,higher,2.0


In [94]:
feature_cols = [
    'insurance_type',
    'prev_readmit_group',
    'los_group',
    'followup_visits_group',
    'risk_score_bin',
    'dc_location',
    'primary_dx_tier',
    'age_bin'
]

In [95]:
df[feature_cols] = df[feature_cols].astype('category')

In [96]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X = df[feature_cols]
y = df['label']

X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)
model.score(X_test, y_test)

0.8225

In [97]:
coef = pd.Series(model.coef_[0], index=X_encoded.columns)

# Sort alphabetically by feature name
coef_sorted_by_name = coef.sort_index()

coef_sorted_by_name

age_bin_2.0                        0.479402
age_bin_3.0                        1.031179
dc_location_Home/Rehab            -0.236001
followup_visits_group_2           -0.033690
followup_visits_group_3           -0.001766
followup_visits_group_4            0.081483
followup_visits_group_5 or more   -0.132139
insurance_type_Private            -0.071024
insurance_type_Uninsured           0.075315
los_group_5                        0.010022
los_group_6 to 8                  -0.050061
los_group_9+                       0.206235
prev_readmit_group_1               0.387341
prev_readmit_group_2               0.763372
primary_dx_tier_lower             -0.299950
risk_score_bin_2                   0.351221
risk_score_bin_3                   0.704581
risk_score_bin_4                   0.950414
risk_score_bin_5                   1.148430
risk_score_bin_6                   1.633764
risk_score_bin_7                   2.589613
dtype: float64

In [98]:
def show_raw_coefs_for(model, feature):
    coefs = pd.Series(model.coef_.flatten(), index=model.feature_names_in_)
    # return coefs[coefs.index.str.startswith(feature + "_")]
    print('baseline = 0.0000')
    print(coefs[coefs.index.str.startswith(feature + "_")])

In [99]:
group = 'followup_visits_group'
bins = df[group].value_counts()
print(bins)
show_raw_coefs_for(model, group)

followup_visits_group
2            2469
4            2307
5 or more    1986
3             992
0 to 1        246
Name: count, dtype: int64
baseline = 0.0000
followup_visits_group_2           -0.033690
followup_visits_group_3           -0.001766
followup_visits_group_4            0.081483
followup_visits_group_5 or more   -0.132139
dtype: float64


In [100]:
df = df.drop(columns=['followup_visits_group'])

In [101]:
group = 'los_group'
bins = df[group].value_counts()
print(bins)
show_raw_coefs_for(model, group)

los_group
6 to 8    4205
9+        2851
5          764
3 to 4     180
Name: count, dtype: int64
baseline = 0.0000
los_group_5         0.010022
los_group_6 to 8   -0.050061
los_group_9+        0.206235
dtype: float64


In [102]:
df['los_group'] = df['los_group'].map({
    '3 to 4': '5 or less',
    '5': '5 or less',
    '6 to 8': '6 to 8',
    '9+': '9+'
})
df['los_group'].value_counts()

los_group
6 to 8       4205
9+           2851
5 or less     944
Name: count, dtype: int64

In [103]:
df.to_pickle('coef_clean_two.pkl')